# SHIVA-LLM — Objective 02, Stage 3
## Multi-objective reward specification and offline transition construction
**Runtime:** Google Colab Python 3, **CPU (None)**; 8–12 GB RAM recommended. No GPU needed.

**Required uploads:** `objective02_stage2c_crosswalk.zip` and `objective02_starter_results.zip` (names may have `(1)` suffix). The original HyperKT **split** is authoritative. This notebook never reuses Stage 1 split assignments.

**Research caution:** Stage 1 `logged_action` is an observationally constructed category, not necessarily a verified instructor intervention. Reward terms below are *proxies*, not causal learning gains. No offline policy-value or treatment-effect claim is supported by this dataset alone.

In [ ]:
import os, zipfile, json, io, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import files
print('Upload both ZIP files (crosswalk and starter results).')
uploaded=files.upload()
paths=list(uploaded)
assert len(paths)>=2, 'Upload the crosswalk ZIP and Stage 1 starter results ZIP.'
def locate(member):
    found=[]
    for p in paths:
        if not zipfile.is_zipfile(p): continue
        with zipfile.ZipFile(p) as z:
            for name in z.namelist():
                if name.endswith(member): found.append((p,name))
    if len(found)!=1: raise ValueError(f'Expected one {member}, found {found}')
    return found[0]
def read_csv_member(member):
    p,name=locate(member)
    with zipfile.ZipFile(p) as z:
        return pd.read_csv(z.open(name),low_memory=False)
cross=read_csv_member('hyperkt_event_crosswalk.csv')
starter=read_csv_member('transitions.csv')
print('Crosswalk',cross.shape,'Stage 1',starter.shape)
print('Crosswalk fields:',cross.columns.tolist())
print('Starter fields:',starter.columns.tolist())

### 1. Structural validation and exact event-key join
The original HyperKT learner partition is retained. Any unmatched events remain visible, rather than receiving invented actions. The notebook does not equate sequence-local positions with original events.

In [ ]:
required_cross={'user_id','order_id','split','window_index','position','target_correct','hyperkt_pred_correct','target_skill'}
required_start={'user_id','order_id','logged_action','prior_attempts','prior_accuracy','split'}
assert required_cross.issubset(cross.columns), required_cross-set(cross.columns)
assert required_start.issubset(starter.columns), required_start-set(starter.columns)
assert not cross.duplicated(['user_id','order_id']).any(), 'Duplicate HyperKT original-event keys'
assert not starter.duplicated(['user_id','order_id']).any(), 'Duplicate Stage 1 original-event keys'
assert not cross.duplicated(['split','window_index','position']).any(), 'Duplicate sequence positions'
assert set(cross.split.unique())=={'train','val','test'}
assert cross.groupby('user_id').split.nunique().max()==1, 'HyperKT split leakage'
assert cross.hyperkt_pred_correct.between(0,1).all()
assert cross.target_correct.isin([0,1]).all()
if 'target_correct_pred' in cross:
    assert (cross.target_correct==cross.target_correct_pred).all(), 'Label alignment mismatch'
if 'target_skill_pred' in cross:
    assert (cross.target_skill==cross.target_skill_pred).all(), 'Skill alignment mismatch'
keep=['user_id','order_id','split','logged_action','prior_attempts','prior_accuracy']
joined=cross.merge(starter[keep],on=['user_id','order_id'],how='left',validate='one_to_one',indicator=True,suffixes=('','_stage1'))
assert len(joined)==len(cross)
joined['has_logged_action']=(joined['_merge']=='both') & joined.logged_action.notna()
joined['split_conflict']=joined.has_logged_action & joined.split.ne(joined.split_stage1)
print('Match counts:',joined['_merge'].value_counts().to_dict())
print('Split conflicts among matched:',int(joined.split_conflict.sum()))
print(pd.crosstab(joined.split,joined.split_stage1,dropna=False))
print('Observed action values:',sorted(joined.logged_action.dropna().unique().tolist()))

### 2. Reward proxies (no future information in state)
- `r_performance`: **observed** correctness of the current target event (reward is observed *after* the action).
- `r_efficiency_proxy`: `1/(1+prior_attempts)`, a bounded **prior-history proxy**, **not** actual instructional cost or time.
- `r_suitability_proxy`: `1 - abs(p_correct - 0.65)/0.65` clipped to [0,1], an illustrative **predicted challenge-match proxy**, not measured question difficulty.

Weights default to 0.6/0.2/0.2. Do not optimize or claim effectiveness based on these arbitrary defaults. `logged_action` categories must be authenticated before meaningful offline policy learning.

In [ ]:
W={'performance':0.60,'efficiency_proxy':0.20,'suitability_proxy':0.20}
assert abs(sum(W.values())-1)<1e-10
joined['r_performance']=joined.target_correct.astype(float)
joined['r_efficiency_proxy']=1.0/(1.0+joined.prior_attempts.clip(lower=0))
joined.loc[joined.prior_attempts.isna(),'r_efficiency_proxy']=np.nan
joined['r_suitability_proxy']=(1-(joined.hyperkt_pred_correct-0.65).abs()/0.65).clip(0,1)
joined['reward_proxy']= (W['performance']*joined.r_performance +
                         W['efficiency_proxy']*joined.r_efficiency_proxy +
                         W['suitability_proxy']*joined.r_suitability_proxy)
print(joined[['r_performance','r_efficiency_proxy','r_suitability_proxy','reward_proxy']].describe().round(4))

### 3. Offline transitions
`state` includes only the **pre-response** HyperKT correctness estimate and prior-history features. The `next_state` is the next *observed* event for the same learner within the authoritative split, not a counterfactual transition. Sequences may have omitted original interactions; these transitions are therefore **observed-event adjacency**, not necessarily consecutive original interactions. Last observed event per learner is marked terminal **in this exported subset only**.

In [ ]:
joined=joined.sort_values(['split','user_id','order_id'],kind='mergesort').reset_index(drop=True)
g=joined.groupby(['split','user_id'],sort=False)
joined['next_order_id']=g.order_id.shift(-1)
joined['next_hyperkt_pred_correct']=g.hyperkt_pred_correct.shift(-1)
joined['next_prior_accuracy']=g.prior_accuracy.shift(-1)
joined['next_prior_attempts']=g.prior_attempts.shift(-1)
joined['subset_terminal']=joined.next_order_id.isna()
joined['observed_event_gap']=joined.next_order_id-joined.order_id
# Only matched rows with valid prior-history features and a defined logged action are candidates.
valid=joined.has_logged_action & joined[['prior_attempts','prior_accuracy','reward_proxy']].notna().all(axis=1)
joined['eligible_for_action_diagnostics']=valid
print('Eligible for action diagnostics:',int(valid.sum()),'/',len(joined))
print('Subset terminal:',int(joined.subset_terminal.sum()))
print('Note: no policy training or causal evaluation is performed.')

### 4. Audit and ZIP export
This notebook exports all aligned rows, plus a restricted matched subset, reward definitions, per-split counts, and reproducibility metadata. It does **not** silently fill missing actions or replace the HyperKT splits.

In [ ]:
out=Path('/content/objective02_stage3');out.mkdir(exist_ok=True)
joined.drop(columns=['_merge']).to_csv(out/'hyperkt_aligned_all_events.csv',index=False)
joined.loc[valid].drop(columns=['_merge']).to_csv(out/'matched_action_diagnostics.csv',index=False)
summary={
 'crosswalk_events':int(len(cross)), 'stage1_events':int(len(starter)),
 'matched_events':int(joined.has_logged_action.sum()),
 'unmatched_crosswalk_events':int((~joined.has_logged_action).sum()),
 'split_conflicts_on_matched_events':int(joined.split_conflict.sum()),
 'eligible_for_action_diagnostics':int(valid.sum()),
 'hyperkt_split_counts':{str(k):int(v) for k,v in joined.split.value_counts().items()},
 'weights':W,
 'reward_components':{'performance':'observed current target correctness',
                      'efficiency_proxy':'1/(1+prior_attempts); prior history only; not actual cost',
                      'suitability_proxy':'clipped proximity of predicted correctness to 0.65; not measured difficulty'},
 'state_warning':'HyperKT correctness probabilities are not full latent states',
 'action_warning':'Stage 1 logged_action is not authenticated as a real instructional intervention',
 'split_policy':'HyperKT split authoritative; Stage 1 split discarded for evaluation',
 'transition_warning':'next observed exported event may skip original interactions; subset_terminal not true episode termination',
 'offline_policy_training_ready':False,
 'causal_policy_evaluation_ready':False
}
with open(out/'stage3_audit.json','w') as f:json.dump(summary,f,indent=2)
pd.DataFrame([{'split':s,'events':len(d),'learners':d.user_id.nunique(),
               'matched_actions':int(d.has_logged_action.sum()),
               'conflicting_old_splits':int(d.split_conflict.sum())}
              for s,d in joined.groupby('split')]).to_csv(out/'split_summary.csv',index=False)
with open(out/'README.txt','w') as f:
 f.write('SHIVA-LLM Objective 02 Stage 3: reward proxy and observational transition preparation.\n'
         'NOT a validated reinforcement-learning policy dataset. See stage3_audit.json.\n')
archive='/content/objective02_stage3_reward_dataset.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in out.iterdir():z.write(p,p.name)
print(json.dumps(summary,indent=2))
print('Created:',archive, 'bytes:',os.path.getsize(archive))
files.download(archive)